# Chapter 16: Operator Overloading

*Fluent Python, 2nd Edition -- Luciano Ramalho*

**Key topics:** Unary operators, `__add__`/`__radd__`, scalar & matrix multiplication, rich comparison operators, augmented assignment operators, Python's operator overloading rules.

---

## 1. Operator Overloading 101

Python's operator overloading lets user-defined types work with infix operators like `+`, `*`, `@`, and comparison operators. Three key constraints:

1. **Cannot redefine** operators for built-in types
2. **Cannot create** new operators -- only overload existing ones
3. **Cannot overload** `is`, `and`, `or`, `not` (but bitwise `&`, `|`, `~` can be)

The golden rule: **operator methods must return new objects**, never mutate the operands (except for augmented assignment on mutable types).

## 2. Unary Operators: `__neg__`, `__pos__`, `__abs__`, `__invert__`

| Operator | Special Method | Description |
|----------|---------------|-------------|
| `-x` | `__neg__` | Arithmetic negation |
| `+x` | `__pos__` | Arithmetic unary plus |
| `~x` | `__invert__` | Bitwise NOT (`~x == -(x+1)`) |
| `abs(x)` | `__abs__` | Absolute value |

**Rule:** Always return a **new object**, never mutate `self`.

In [ ]:
import math
import itertools
from array import array


class Vector:
    """A multidimensional Vector class demonstrating operator overloading."""
    typecode = 'd'

    def __init__(self, components):
        self._components = array(self.typecode, components)

    def __iter__(self):
        return iter(self._components)

    def __len__(self):
        return len(self._components)

    def __repr__(self):
        components = ', '.join(repr(c) for c in self._components)
        return f'Vector([{components}])'

    def __eq__(self, other):
        if isinstance(other, Vector):
            return (len(self) == len(other) and
                    all(a == b for a, b in zip(self, other)))
        return NotImplemented

    # --- Unary operators ---
    def __abs__(self):
        return math.hypot(*self)

    def __neg__(self):
        return Vector(-x for x in self)

    def __pos__(self):
        return Vector(self)  # returns a new copy


v = Vector([1, 2, 3])
print(f"v       = {v}")
print(f"-v      = {-v}")
print(f"+v      = {+v}")
print(f"abs(v)  = {abs(v):.4f}")
print(f"-v is v: {-v is v}")  # always a new object

### When `x != +x`

Two cases in the standard library where `x != +x`:

1. **`decimal.Decimal`** -- if the arithmetic context precision changes between creating `x` and evaluating `+x`
2. **`collections.Counter`** -- `+counter` drops zero and negative tallies

In [ ]:
import decimal
from collections import Counter

# Case 1: Decimal with changed precision
ctx = decimal.getcontext()
ctx.prec = 40
one_third = decimal.Decimal('1') / decimal.Decimal('3')
print(f"Precision 40: {one_third}")
print(f"x == +x (prec 40): {one_third == +one_third}")

ctx.prec = 28  # change precision
print(f"x == +x (prec 28): {one_third == +one_third}")
print(f"+x (prec 28):      {+one_third}")

# Case 2: Counter with negative tallies
ct = Counter('abracadabra')
ct['r'] = -3
ct['d'] = 0
print(f"\nCounter:  {ct}")
print(f"+Counter: {+ct}")  # drops zero and negative values

## 3. Overloading `+` with `__add__` and `__radd__`

For `a + b`, Python follows this dispatch protocol:

1. Call `a.__add__(b)` -- if it returns `NotImplemented`, go to step 2
2. Call `b.__radd__(a)` -- if it returns `NotImplemented`, go to step 3
3. Raise `TypeError`

**Critical distinction:** Return `NotImplemented` (the singleton), do NOT raise `NotImplementedError` (the exception). Returning `NotImplemented` gives the other operand a chance to handle the operation.

In [ ]:
class Vector:
    typecode = 'd'

    def __init__(self, components):
        self._components = array('d', components)

    def __iter__(self):
        return iter(self._components)

    def __len__(self):
        return len(self._components)

    def __repr__(self):
        components = ', '.join(repr(c) for c in self._components)
        return f'Vector([{components}])'

    def __eq__(self, other):
        if isinstance(other, Vector):
            return (len(self) == len(other) and
                    all(a == b for a, b in zip(self, other)))
        return NotImplemented

    def __abs__(self):
        return math.hypot(*self)

    def __neg__(self):
        return Vector(-x for x in self)

    def __pos__(self):
        return Vector(self)

    # --- __add__ with proper error handling ---
    def __add__(self, other):
        try:
            pairs = itertools.zip_longest(self, other, fillvalue=0.0)
            return Vector(a + b for a, b in pairs)
        except TypeError:
            return NotImplemented

    # --- __radd__ delegates to __add__ (addition is commutative) ---
    def __radd__(self, other):
        return self + other


v1 = Vector([3, 4, 5])
v2 = Vector([6, 7, 8])

# Vector + Vector
print(f"v1 + v2     = {v1 + v2}")

# Vectors of different lengths (shorter one padded with 0.0)
v3 = Vector([1, 2])
print(f"v1 + v3     = {v1 + v3}")

# Vector + tuple (works via duck typing -- tuple is iterable)
print(f"v1 + (10,20,30) = {v1 + (10, 20, 30)}")

# tuple + Vector (triggers __radd__)
print(f"(10,20,30) + v1 = {(10, 20, 30) + v1}")

### Why `__radd__` matters

Without `__radd__`, reversed operations fail:

```python
(10, 20, 30) + v1  # tuple.__add__ doesn't know about Vector
                    # Without __radd__, this raises TypeError
```

Since `+` is commutative for our vectors, `__radd__` simply delegates to `__add__`. For non-commutative operators, you would need different logic.

In [ ]:
# Demonstrating the dispatch: what happens with incompatible types
try:
    result = v1 + "ABC"
except TypeError as e:
    print(f"v1 + 'ABC' -> TypeError: {e}")

try:
    result = v1 + 42
except TypeError as e:
    print(f"v1 + 42   -> TypeError: {e}")

## 4. Overloading `*` and `@` for Scalar and Matrix Multiplication

### Scalar multiplication with `__mul__` / `__rmul__`

Uses **duck typing**: try to convert the scalar to `float`; if that fails, return `NotImplemented`.

### Dot product with `__matmul__` / `__rmatmul__` (the `@` operator)

Uses **goose typing**: check `isinstance(other, abc.Sized) and isinstance(other, abc.Iterable)` to ensure the operand supports both `len()` and iteration.

The `@` operator was added in Python 3.5 via PEP 465 for matrix multiplication.

In [ ]:
from collections import abc


class Vector:
    typecode = 'd'

    def __init__(self, components):
        self._components = array('d', components)

    def __iter__(self):
        return iter(self._components)

    def __len__(self):
        return len(self._components)

    def __repr__(self):
        components = ', '.join(repr(c) for c in self._components)
        return f'Vector([{components}])'

    def __eq__(self, other):
        if isinstance(other, Vector):
            return (len(self) == len(other) and
                    all(a == b for a, b in zip(self, other)))
        return NotImplemented

    def __abs__(self):
        return math.hypot(*self)

    def __neg__(self):
        return Vector(-x for x in self)

    def __pos__(self):
        return Vector(self)

    def __add__(self, other):
        try:
            pairs = itertools.zip_longest(self, other, fillvalue=0.0)
            return Vector(a + b for a, b in pairs)
        except TypeError:
            return NotImplemented

    def __radd__(self, other):
        return self + other

    # --- Scalar multiplication (duck typing) ---
    def __mul__(self, scalar):
        try:
            factor = float(scalar)
        except TypeError:
            return NotImplemented
        return Vector(n * factor for n in self)

    def __rmul__(self, scalar):
        return self * scalar

    # --- Dot product / matrix multiplication (goose typing) ---
    def __matmul__(self, other):
        if (isinstance(other, abc.Sized) and
                isinstance(other, abc.Iterable)):
            if len(self) == len(other):
                return sum(a * b for a, b in zip(self, other))
            else:
                raise ValueError('@ requires vectors of equal length.')
        return NotImplemented

    def __rmatmul__(self, other):
        return self @ other


va = Vector([1, 2, 3])

# Scalar multiplication
print(f"va * 10     = {va * 10}")
print(f"11 * va     = {11 * va}")
print(f"va * True   = {va * True}")

from fractions import Fraction
print(f"va * 1/3    = {va * Fraction(1, 3)}")

# Dot product with @
vb = Vector([5, 6, 7])
print(f"\nva @ vb     = {va @ vb}")    # 1*5 + 2*6 + 3*7 = 38
print(f"[10,20,30] @ vb = {[10, 20, 30] @ vb}")  # goose typing in action

In [ ]:
# @ operator with incompatible types
try:
    result = va @ 3
except TypeError as e:
    print(f"va @ 3 -> TypeError: {e}")

try:
    result = va @ Vector([1, 2])  # different lengths
except ValueError as e:
    print(f"va @ Vector([1,2]) -> ValueError: {e}")

## 5. Rich Comparison Operators

Rich comparison operators have special dispatch rules:

| Expression | Forward call | Reverse call | Fallback |
|-----------|-------------|-------------|----------|
| `a == b` | `a.__eq__(b)` | `b.__eq__(a)` | `id(a) == id(b)` |
| `a != b` | `a.__ne__(b)` | `b.__ne__(a)` | `not (a == b)` |
| `a > b` | `a.__gt__(b)` | `b.__lt__(a)` | Raise `TypeError` |
| `a < b` | `a.__lt__(b)` | `b.__gt__(a)` | Raise `TypeError` |
| `a >= b` | `a.__ge__(b)` | `b.__le__(a)` | Raise `TypeError` |
| `a <= b` | `a.__le__(b)` | `b.__ge__(a)` | Raise `TypeError` |

Key differences from arithmetic operators:
- **No separate "r" methods** -- the reverse of `__gt__` is `__lt__` (not `__rgt__`)
- **`==` falls back to identity** comparison (object IDs) instead of raising `TypeError`
- **`!=` negates `==`** -- you almost never need to implement `__ne__`

In [ ]:
class Vector:
    typecode = 'd'

    def __init__(self, components):
        self._components = array('d', components)

    def __iter__(self):
        return iter(self._components)

    def __len__(self):
        return len(self._components)

    def __repr__(self):
        components = ', '.join(repr(c) for c in self._components)
        return f'Vector([{components}])'

    # Improved __eq__: only accept Vector instances
    def __eq__(self, other):
        if isinstance(other, Vector):
            return (len(self) == len(other) and
                    all(a == b for a, b in zip(self, other)))
        return NotImplemented  # let the other side try


va = Vector([1.0, 2.0, 3.0])
vb = Vector([1, 2, 3])

print(f"va == vb:       {va == vb}")      # True (same components)
print(f"va != vb:       {va != vb}")      # False (__ne__ from object negates __eq__)
print(f"va == (1,2,3):  {va == (1, 2, 3)}")  # False (tuple != Vector)
print(f"va == 'xyz':    {va == 'xyz'}")    # False (falls back to id comparison)

### How `va == (1, 2, 3)` returns `False`

1. Python calls `Vector.__eq__(va, (1,2,3))`
2. Our `__eq__` sees `(1,2,3)` is not a `Vector`, returns `NotImplemented`
3. Python tries `tuple.__eq__((1,2,3), va)`
4. `tuple.__eq__` does not know about `Vector`, returns `NotImplemented`
5. As a fallback for `==`, Python compares `id(va) == id((1,2,3))` which is `False`

In [ ]:
# functools.total_ordering: auto-generate missing comparison methods
from functools import total_ordering


@total_ordering
class Temperature:
    """Only define __eq__ and __lt__; total_ordering fills the rest."""

    def __init__(self, value):
        self.value = float(value)

    def __repr__(self):
        return f"Temperature({self.value})"

    def __eq__(self, other):
        if isinstance(other, Temperature):
            return self.value == other.value
        return NotImplemented

    def __lt__(self, other):
        if isinstance(other, Temperature):
            return self.value < other.value
        return NotImplemented


t1 = Temperature(100)
t2 = Temperature(212)

# total_ordering generates __gt__, __ge__, __le__ from __eq__ + __lt__
print(f"t1 < t2:  {t1 < t2}")
print(f"t1 > t2:  {t1 > t2}")
print(f"t1 >= t1: {t1 >= t1}")
print(f"t1 <= t2: {t1 <= t2}")
print(f"t1 != t2: {t1 != t2}")

## 6. Augmented Assignment Operators: `__iadd__`, `__imul__`, etc.

**Without `__iadd__`:** `a += b` desugars to `a = a.__add__(b)` -- creates a **new** object and rebinds the variable. This is the correct behavior for **immutable** types.

**With `__iadd__`:** `a += b` calls `a.__iadd__(b)` -- should modify `self` **in-place** and return `self`. Only for **mutable** types.

Key insight: `+=` can be **more liberal** than `+` in accepted types. Example: `list += any_iterable` works, but `list + non_list` raises `TypeError`.

In [ ]:
# Immutable type: += creates a new object (no __iadd__ needed)
v1 = Vector([1, 2, 3])
v1_id = id(v1)
v1 += Vector([4, 5, 6])  # desugars to: v1 = v1 + Vector([4, 5, 6])
print(f"After +=: {v1}")
print(f"Same object? {id(v1) == v1_id}")  # False -- new object created

In [ ]:
# Mutable type: implementing __iadd__ for in-place modification
import random


class BingoCage:
    """A simple mutable container that picks random items."""

    def __init__(self, items):
        self._items = list(items)
        random.shuffle(self._items)

    def pick(self):
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError("pick from empty BingoCage")

    def inspect(self):
        return tuple(sorted(self._items))

    def __repr__(self):
        return f"BingoCage({self.inspect()})"


class AddableBingoCage(BingoCage):
    """BingoCage with + and += support."""

    def __add__(self, other):
        # + is strict: only accept same type
        if isinstance(other, AddableBingoCage):
            return AddableBingoCage(self.inspect() + other.inspect())
        return NotImplemented

    def __iadd__(self, other):
        # += is liberal: accept any iterable
        if isinstance(other, AddableBingoCage):
            other_iterable = other.inspect()
        else:
            try:
                other_iterable = iter(other)
            except TypeError:
                raise TypeError(
                    "right operand in += must be "
                    "'AddableBingoCage' or an iterable"
                )
        self._items.extend(other_iterable)
        random.shuffle(self._items)
        return self  # MUST return self for in-place operation


# Demonstrate + (creates new)
globe = AddableBingoCage('AEIOU')
globe2 = AddableBingoCage('XYZ')
globe3 = globe + globe2
print(f"globe + globe2 = {globe3}")
print(f"globe unchanged: {globe}")

# Demonstrate += (modifies in-place)
globe_ref = globe
globe += globe2
print(f"\nAfter globe += globe2: {globe}")
print(f"Same object? {globe is globe_ref}")  # True -- modified in-place

# += accepts any iterable (more liberal than +)
globe += ['M', 'N']
print(f"After globe += ['M','N']: {globe}")

In [ ]:
# The contrast: + is strict, += is liberal
try:
    result = globe + [10, 20]  # + only accepts AddableBingoCage
except TypeError as e:
    print(f"globe + [10,20] -> TypeError: {e}")

# But += works with any iterable
globe += [10, 20]  # This is fine!
print(f"globe += [10,20] works: {globe}")

### The `list` built-in follows the same pattern

`list.__add__` only accepts another `list`, but `list.__iadd__` (i.e., `+=`) accepts any iterable -- just like `list.extend()`.

In [ ]:
my_list = [1, 2, 3]

# list + only works with another list
try:
    result = my_list + (4, 5)
except TypeError as e:
    print(f"list + tuple: TypeError: {e}")

# list += works with any iterable
my_list += (4, 5)
print(f"list += tuple: {my_list}")

my_list += range(6, 9)
print(f"list += range: {my_list}")

# Even works with generators!
my_list += (x**2 for x in range(3))
print(f"list += genexp: {my_list}")

## 7. Quick Reference: Infix Operator Method Names

| Operator | Forward | Reverse | In-place | Description |
|----------|---------|---------|----------|-------------|
| `+` | `__add__` | `__radd__` | `__iadd__` | Addition / concatenation |
| `-` | `__sub__` | `__rsub__` | `__isub__` | Subtraction |
| `*` | `__mul__` | `__rmul__` | `__imul__` | Multiplication / repetition |
| `/` | `__truediv__` | `__rtruediv__` | `__itruediv__` | True division |
| `//` | `__floordiv__` | `__rfloordiv__` | `__ifloordiv__` | Floor division |
| `%` | `__mod__` | `__rmod__` | `__imod__` | Modulo |
| `**` | `__pow__` | `__rpow__` | `__ipow__` | Exponentiation |
| `@` | `__matmul__` | `__rmatmul__` | `__imatmul__` | Matrix multiplication |
| `&` | `__and__` | `__rand__` | `__iand__` | Bitwise AND |
| `\|` | `__or__` | `__ror__` | `__ior__` | Bitwise OR |
| `^` | `__xor__` | `__rxor__` | `__ixor__` | Bitwise XOR |
| `<<` | `__lshift__` | `__rlshift__` | `__ilshift__` | Bitwise shift left |
| `>>` | `__rshift__` | `__rrshift__` | `__irshift__` | Bitwise shift right |

## 8. Best Practices Summary

1. **Return `NotImplemented`** (not raise `TypeError`) when you cannot handle an operand -- give the other side a chance
2. **Never mutate operands** in non-augmented operators (`+`, `*`, etc.) -- always return new objects
3. **Use duck typing or goose typing** for type checks -- avoid rigid `isinstance` checks against concrete types
4. **For commutative operators**, `__radd__` can simply delegate to `__add__`
5. **Augmented assignment** (`__iadd__`) should modify `self` in-place and return `self` for mutable types
6. **`+=` can be more liberal** than `+` in accepted types
7. **`__ne__` is free** -- it inherits from `object` and negates `__eq__`
8. **`functools.total_ordering`** generates missing comparison methods from `__eq__` + one ordering method

---
## Exercises

### Exercise 1: Implement `__sub__` and `__rsub__`

Add vector subtraction to the `Vector` class below. Follow the same pattern as `__add__`/`__radd__`.

In [ ]:
class Vector:
    typecode = 'd'

    def __init__(self, components):
        self._components = array('d', components)

    def __iter__(self):
        return iter(self._components)

    def __len__(self):
        return len(self._components)

    def __repr__(self):
        components = ', '.join(repr(c) for c in self._components)
        return f'Vector([{components}])'

    # TODO: Implement __sub__ and __rsub__
    def __sub__(self, other):
        try:
            pairs = itertools.zip_longest(self, other, fillvalue=0.0)
            return Vector(a - b for a, b in pairs)
        except TypeError:
            return NotImplemented

    def __rsub__(self, other):
        # NOTE: subtraction is NOT commutative, so we can't just do self - other
        try:
            pairs = itertools.zip_longest(other, self, fillvalue=0.0)
            return Vector(a - b for a, b in pairs)
        except TypeError:
            return NotImplemented


v1 = Vector([10, 20, 30])
v2 = Vector([1, 2, 3])

print(f"v1 - v2       = {v1 - v2}")
print(f"v1 - (1,2,3)  = {v1 - (1, 2, 3)}")
print(f"(100,200) - v2 = {(100, 200) - v2}")  # uses __rsub__

### Exercise 2: Why does `__rsub__` need different logic?

Unlike `__radd__`, `__rsub__` cannot simply delegate to `__sub__` because subtraction is **not commutative**: `a - b != b - a`. When Python calls `b.__rsub__(a)`, the operands are swapped, so we must compute `other - self` (not `self - other`).

### Exercise 3: Build a `Money` class

Try building a simple `Money` class that supports `+`, `*` (by scalar), and comparison. Think about: what should `Money('USD', 10) + Money('EUR', 5)` do?

In [ ]:
class Money:
    """Simple money class demonstrating operator overloading with type safety."""

    def __init__(self, currency: str, amount: float):
        self.currency = currency
        self.amount = float(amount)

    def __repr__(self):
        return f"Money('{self.currency}', {self.amount:.2f})"

    def __add__(self, other):
        if isinstance(other, Money):
            if self.currency != other.currency:
                raise ValueError(
                    f"Cannot add {self.currency} and {other.currency}")
            return Money(self.currency, self.amount + other.amount)
        return NotImplemented

    def __radd__(self, other):
        return self + other  # commutative

    def __mul__(self, scalar):
        try:
            factor = float(scalar)
        except TypeError:
            return NotImplemented
        return Money(self.currency, self.amount * factor)

    def __rmul__(self, scalar):
        return self * scalar

    def __eq__(self, other):
        if isinstance(other, Money):
            return self.currency == other.currency and self.amount == other.amount
        return NotImplemented


usd10 = Money('USD', 10)
usd20 = Money('USD', 20)

print(f"{usd10} + {usd20} = {usd10 + usd20}")
print(f"{usd10} * 3 = {usd10 * 3}")
print(f"5 * {usd10} = {5 * usd10}")
print(f"{usd10} == {usd10} : {usd10 == Money('USD', 10)}")

try:
    eur = Money('EUR', 5)
    result = usd10 + eur
except ValueError as e:
    print(f"Mixed currencies: {e}")